In [3]:
import Pkg
Pkg.add("JuMP")
Pkg.add("HiGHS")
Pkg.add("Gurobi")


   Resolving package versions...
     Project No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
   Installed Gurobi_jll ─ v13.0.0
   Installed Gurobi ───── v1.9.1
  Installing 1 artifacts
   Installed artifact Gurobi      61.6 MiB
    Updating `C:\Users\huste\.julia\environments\v1.12\Project.toml`
  [2e9cd046] + Gurobi v1.9.1
    Updating `C:\Users\huste\.julia\environments\v1.12\Manifest.toml`
  [2e9cd046] + Gurobi v1.9.1
  [c018c7e6] + Gurobi_jll v13.0.0
    Building Gurobi → `C:\Users\huste\.julia\scratchspaces\44cfe95a-1eb2-52ea-b672-e2afdf69b78f\7e700e7cec5c7

# Blending 2

In [ ]:
# agiso@dtu.dk
using JuMP, HiGHS

##### ----- variable ----- #####
M = 6
I = 5

months = 1:M
indencies = 1:I

# Make a matrix with cost
cost = [
    110 120 130 110 115;
    130 130 110 90 115;
    110 140 130 100 95;
    120 110 120 120 125;
    100 120 150 110 105;
    90 100 140 80 135
]

earnings = 150
storage_cost = 5

# Production limit
prod_limit_1 = 200
prod_limit_2 = 250

# Hardness
hardeness = [8.8, 6.1, 2.0, 4.2, 5.0]
h_lower = 3
h_upper = 6

# storage
s_start = 500
s_end = 500
s_high = 1000

##### ----- Model ----- #####
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

##### ----- Variables ----- #####
@variable(model, p[months, indencies] >= 0)
@variable(model, s[months, indencies] >= 0)
@variable(model, sale[months, indencies] >= 0)

##### ----- objectives ----- #####
@objective(model, Max,
    sum(earnings * sale[m, i] for m in months, i in indencies) -
    sum(cost[m, i] * p[m, i] for m in months, i in indencies) -
    sum(storage_cost * s[m, i] for m in months, i in indencies)
)

##### ----- Simple constraints ----- #####
# Production capacity constraints
@constraint(model, [m in months],
    sale[m, 1] + sale[m, 2] <= prod_limit_1
)
@constraint(model, [m in months],
    sale[m, 3] + sale[m, 4] + sale[m, 5] <= prod_limit_2
)

# Final storage constraints
@constraint(model, [i in indencies],
    s[6, i] == s_end
)
# Storage capacity constraints
@constraint(model, [m in months, i in indencies],
    s[m, i] <= s_high
)

#### ----- Constraints ----- #####
# We store what we do not sell + storage from last month
@constraint(model, [m in months, i in indencies],
    s[m, i] == (m > 1 ? s[m-1, i] : s_start) + p[m, i] - sale[m, i]
)

# Hardness constraints
@constraint(model, [m in months],
    sum(hardeness[i] * sale[m,i] for i in indencies) >= h_lower * sum(sale[m,i] for i in indencies)
)
@constraint(model, [m in months],
    sum(hardeness[i] * sale[m,i] for i in indencies) <= h_upper * sum(sale[m,i] for i in indencies)
)

optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("\nProduction:")
println(value.(p))

println("\nSales:")
println(value.(sale))

println("\nStorage:")
println(value.(s))


LoadError: Gurobi Error 10009: No Gurobi license found (user huste, host ALX26, hostid fa9b4a17, cores 24)

# 

# Factory planning

In [18]:
# 7 products 1-7
# Each with different manufacturing processes (grinding, vertical drilling, horizontal drilling, boring, or planing)
# The factory has four grinders, two vertical drills, three horizontal drills,one borer, and one planer.

# Indencies
products = 1:7
processes = 1:5
months = 1:6
M = length(months)
P = length(products)

# Variables
storage_cost = 0.5
storage_init = 0
storage_end = 50


month_days = 24
working_hours = month_days * 8 * 2 # 2 shifts
machines = [4, 2, 3, 1, 1] # number of machines per process
machines_avil = [machines[pr] for m in months, pr in processes]

# Removeing machines for maintenance
# January
machines_avil[1, 1] -= 1 # grinder
machines_avil[2, 3] -= 2 # horizontal drills
machines_avil[3, 4] -= 1 # boring
machines_avil[4, 2] -= 1 # vertical drilling
machines_avil[5, 1] -= 1 # grinder
machines_avil[5, 2] -= 1 # vertical drilling
machines_avil[6, 5] -= 1 # planing
machines_avil[6, 3] -= 1 # horizontal drilling

# Data
demand = [
    500 1000 300 300 800 200 100;
    600 500  200 0   400 300 150;
    300 600  0   0   500 400 100;
    200 300  400 500 200 0   100;
    0   100  500 100 1000 300 0;
    500 500  100 300 1100 500 60
];

# process hours per product
process_hours = [
    0.50 0.70 0.00 0.00 0.30 0.20 0.50;  # Grinding
    0.10 0.20 0.00 0.30 0.00 0.60 0.00;  # Vertical drilling
    0.20 0.00 0.80 0.00 0.00 0.00 0.60;  # Horizontal drilling
    0.05 0.03 0.00 0.07 0.10 0.00 0.08;  # Boring
    0.00 0.00 0.01 0.00 0.05 0.00 0.05;  # Planing
];

profits = [10, 6, 8, 4, 11, 9, 3]

########## ---------- model ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)


########## ---------- Variables ---------- ##########
@variable(model, production[months, products] >= 0)
@variable(model, storage[months, products] >= 0)
@variable(model, profit[months, products] >= 0)

########## ---------- Constraints ---------- ##########
# Demand constraints
@constraint(model, [m in months, p in products],
    profit[m, p] <= demand[m, p]
)

########## ---------- Constraints - Storage ---------- ##########
# Storage is equal to the previous month + production - profit
@constraint(model, [m in months, i in products],
    storage[m, i] == (m > 1 ? storage[m-1, i] : storage_init) + production[m, i] - profit[m, i]
)

# Final storage constraints
@constraint(model, [i in products],
    storage[M, i] == storage_end
)

# Storage capacity constraints
@constraint(model, [m in months, i in products],
    storage[m, i] <= 100
)

########## ---------- Constraints - Production ---------- ##########
# If we want to produce product i we need to use the processes in column i of process_hours
    # st. if we make 1 untit of prod_1 whe need process_hours[1,1] hours of grinding, process_hours[2,1] hours of vertical drilling, etc.
@constraint(model, [m in months, pr in processes],
    sum(process_hours[pr, p] * production[m, p] for p in products) <= working_hours * machines_avil[m, pr]
)


########## ---------- Objectives ---------- ##########
@objective(model, Max,
    sum(profits[p] * profit[m, p] for m in months, p in products) -
    sum(storage_cost * storage[m, p] for m in months, p in products)
)

optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))


println("\nProduction:")
println(value.(production))

println("\nSales:")
println(value.(profit))

println("\nStorage:")
println(value.(storage))

Optimal solution:
z = 93715.17857142858

Production:
2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, 1:6
    Dimension 2, 1:7
And data, a 6×7 Matrix{Float64}:
 500.0  888.5714285714287  382.5  300.0   800.0  200.0   -0.0
 700.0  600.0              117.5   -0.0   500.0  300.0  250.0
  -0.0    0.0                0.0    0.0     0.0  400.0    0.0
 200.0  300.0              400.0  500.0   200.0   -0.0  100.0
   0.0  100.0              600.0  100.0  1100.0  300.0  100.0
 550.0  550.0               -0.0  350.0     0.0  550.0    0.0

Sales:
2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, 1:6
    Dimension 2, 1:7
And data, a 6×7 Matrix{Float64}:
 500.0  888.5714285714287  300.0  300.0   800.0  200.0    0.0
 600.0  500.0              200.0   -0.0   400.0  300.0  150.0
 100.0  100.0               -0.0   -0.0   100.0  400.0  100.0
 200.0  300.0              400.0  500.0   200.0   -0.0  100.0
  -0.0  100.0              500.0  100.0  1000.0

# Shed planning

In [39]:
tasks = 1:13
time = [
    1, 3, 5, 3, 7, 4, 3, 1, 1, 1, 3, 1, 0
]

# Dependencies
required = falses(13, 13)
required[2, 1] = true
required[3, 2] = true
required[4, 2] = true
required[5, 3] = true
required[5, 4] = true
required[6, 3] = true
required[6, 4] = true
required[7, 6] = true
required[8, 7] = true
required[9, 7] = true
required[10, 9] = true
required[11, 9] = true
required[12, 9] = true
required[13, 5] = true
required[13, 8] = true
required[13, 10] = true
required[13, 11] = true
required[13, 12] = true

########## ---------- model ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

########## ---------- Variables ---------- ##########
@variable(model, start_time[tasks] >= 0)

########## ---------- Constraints ---------- ##########
# Dependency constraints
for i in tasks, j in tasks
    if required[i, j] == true
        @constraint(model,
            start_time[i] >= start_time[j] + time[j]
        )
    end
end

@objective(model, Min,
    start_time[13]
)

optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))
println("start_time = ", value.(start_time))

Optimal solution:
z = 20.0
start_time = 1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, 1:13
And data, a 13-element Vector{Float64}:
  0.0
  1.0
  4.0
  4.0
  9.0
  9.0
 13.0
 16.0
 16.0
 17.0
 17.0
 17.0
 20.0


# Blending 2

In [ ]:
# agiso@dtu.dk
using JuMP, HiGHS

##### ----- variable ----- #####
M = 6
I = 5

months = 1:M
indencies = 1:I

# Make a matrix with cost
cost = [
    110 120 130 110 115;
    130 130 110 90 115;
    110 140 130 100 95;
    120 110 120 120 125;
    100 120 150 110 105;
    90 100 140 80 135
]

earnings = 150
storage_cost = 5

# Production limit
prod_limit_1 = 200
prod_limit_2 = 250

# Hardness
hardeness = [8.8, 6.1, 2.0, 4.2, 5.0]
h_lower = 3
h_upper = 6

# storage
s_start = 500
s_end = 500
s_high = 1000

##### ----- Model ----- #####
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

##### ----- Variables ----- #####
@variable(model, p[months, indencies] >= 0)
@variable(model, s[months, indencies] >= 0)
@variable(model, sale[months, indencies] >= 0)

##### ----- objectives ----- #####
@objective(model, Max,
    sum(earnings * sale[m, i] for m in months, i in indencies) -
    sum(cost[m, i] * p[m, i] for m in months, i in indencies) -
    sum(storage_cost * s[m, i] for m in months, i in indencies)
)

##### ----- Simple constraints ----- #####
# Production capacity constraints
@constraint(model, [m in months],
    sale[m, 1] + sale[m, 2] <= prod_limit_1
)
@constraint(model, [m in months],
    sale[m, 3] + sale[m, 4] + sale[m, 5] <= prod_limit_2
)

# Final storage constraints
@constraint(model, [i in indencies],
    s[6, i] == s_end
)
# Storage capacity constraints
@constraint(model, [m in months, i in indencies],
    s[m, i] <= s_high
)

#### ----- Constraints ----- #####
# We store what we do not sell + storage from last month
@constraint(model, [m in months, i in indencies],
    s[m, i] == (m > 1 ? s[m-1, i] : s_start) + p[m, i] - sale[m, i]
)

# Hardness constraints
@constraint(model, [m in months],
    sum(hardeness[i] * sale[m,i] for i in indencies) >= h_lower * sum(sale[m,i] for i in indencies)
)
@constraint(model, [m in months],
    sum(hardeness[i] * sale[m,i] for i in indencies) <= h_upper * sum(sale[m,i] for i in indencies)
)

optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("\nProduction:")
println(value.(p))

println("\nSales:")
println(value.(sale))

println("\nStorage:")
println(value.(s))
